In [ ]:
import pandas as pd

cols = ["eventid", "iyear", "imonth", "iday", "country_txt", "provstate",
        "city", "latitude", "longitude", "gname", "attacktype1_txt",
        "targtype1_txt", "weaptype1_txt", "nkill", "nwound", "summary"]
df = pd.read_parquet("../data/processed/gtd_full.parquet", columns=cols)

pk = df[df["country_txt"].eq("Pakistan")]
print("Pakistan events:", len(pk))

In [ ]:
city = pk["city"].fillna("MISSING")
print(city[city.str.contains("karachi", case=False)].value_counts())

In [ ]:
karachi = pk[pk["city"].fillna("").str.contains("karachi", case=False)]
print("Karachi events:", len(karachi))

sindh = pk[pk["provstate"].eq("Sindh").fillna(False)]
print("Sindh events:", len(sindh))

print(karachi["iyear"].value_counts().sort_index())

In [ ]:
other_sindh = sindh[~sindh.index.isin(karachi.index)]
print(len(other_sindh))
print(other_sindh["city"].fillna("MISSING").value_counts().head(15))

In [ ]:
print(karachi["provstate"].value_counts(dropna=False))

In [ ]:
box = pk[(pk["latitude"].between(24.7, 25.2)) & (pk["longitude"].between(66.8, 67.4))]
extra = box[~box.index.isin(karachi.index)]
print(len(extra))
print(extra["city"].fillna("MISSING").value_counts().head(15))

In [ ]:
print(karachi["iyear"].value_counts().sort_index().to_string())

k = karachi.assign(
    decade=(karachi["iyear"] // 10) * 10,
    has_summary=karachi["summary"].notna(),
    known_group=~karachi["gname"].eq("Unknown").fillna(True),
)
print(k.groupby("decade").agg(
    events=("eventid", "count"),
    summary=("has_summary", "mean"),
    known_group=("known_group", "mean"),
).round(2))

In [ ]:
in_box = pk["latitude"].between(24.7, 25.2) & pk["longitude"].between(66.8, 67.4)
in_sindh = pk["provstate"].eq("Sindh").fillna(False)
labelled = pk["city"].fillna("").str.contains("karachi", case=False)

scope = pk[labelled | (in_box & in_sindh)]
print("Karachi scope:", len(scope), "| added beyond the city label:", len(scope) - labelled.sum())
print(scope["city"].value_counts().head(12))

In [ ]:
rest = sindh[~sindh.index.isin(scope.index)]
no_coords = rest[rest["latitude"].isna()]
print(len(no_coords))
print(no_coords["city"].fillna("MISSING").value_counts().to_string())